In [6]:
import pandas as pd
import os

def convert_v2_to_v1(input_path, output_path=None):
    if output_path is None:
        base, ext = os.path.splitext(input_path)
        output_path = base + "_v1" + ext

    df = pd.read_csv(input_path, sep='\t', low_memory=False)
    sample_name = os.path.basename(input_path).replace('.tsv', '')

    v1_columns = [
        "sample_name","species","locus","product_subtype","kit_pool","sku",
        "test_name","sample_catalog_tags","sample_rich_tags","sample_rich_tags_json",
        "hla_class_i","hla_class_ii","kit_control","total_templates",
        "productive_templates","outofframe_templates","stop_templates","dj_templates",
        "total_rearrangements","productive_rearrangements","outofframe_rearrangements",
        "stop_rearrangements","dj_rearrangements","total_reads",
        "total_productive_reads","total_outofframe_reads","total_stop_reads",
        "total_dj_reads","productive_simpson_clonality","productive_clonality",
        "productive_entropy","sample_simpson_clonality","sample_clonality",
        "sample_entropy","sample_amount_ng","sample_cells_mass_estimate",
        "fraction_productive_of_cells_mass_estimate","sample_cells",
        "fraction_productive_of_cells","max_productive_frequency","max_frequency",
        "counting_method","primer_set","sequence_result_status","release_date",
        "upload_date","sample_tags","fraction_productive","order_name","kit_id",
        "total_t_cells","total_templates_agg",
        "rearrangement","amino_acid","frame_type","rearrangement_type",
        "templates","seq_reads","frequency","productive_frequency",
        "cdr3_length","v_family","v_gene","v_allele",
        "d_family","d_gene","d_allele","j_family","j_gene","j_allele",
        "v_deletions","d5_deletions","d3_deletions","j_deletions",
        "n2_insertions","n1_insertions","v_index","n1_index","n2_index",
        "d_index","j_index",
        "v_family_ties","v_gene_ties","v_allele_ties",
        "d_family_ties","d_gene_ties","d_allele_ties",
        "j_family_ties","j_gene_ties","j_allele_ties",
        "sequence_tags","v_shm_count","v_shm_indexes","antibody",
        "bio_identity","rearrangement_trunc",
        "v_resolved","d_resolved","j_resolved",
        "extended_rearrangement",
        "cdr1_rearrangement","cdr1_amino_acid","cdr1_start_index","cdr1_rearrangement_length",
        "cdr2_rearrangement","cdr2_amino_acid","cdr2_start_index","cdr2_rearrangement_length",
        "cdr3_rearrangement","cdr3_amino_acid","cdr3_start_index","cdr3_rearrangement_length",
        "chosen_v_family","chosen_v_gene","chosen_v_allele",
        "chosen_j_family","chosen_j_gene","chosen_j_allele",
    ]

    rename_map = {
        "nucleotide": "rearrangement",
        "template_frequency": "frequency",
        "productive_template_frequency": "productive_frequency",
        "nucleotide_extended": "extended_rearrangement",
        "cdr1_nucleotide": "cdr1_rearrangement",
        "cdr1_index": "cdr1_start_index",
        "cdr1_nucleotide_length": "cdr1_rearrangement_length",
        "cdr2_nucleotide": "cdr2_rearrangement",
        "cdr2_index": "cdr2_start_index",
        "cdr2_nucleotide_length": "cdr2_rearrangement_length",
        "cdr3_nucleotide": "cdr3_rearrangement",
        "cdr3_index": "cdr3_start_index",
        "cdr3_nucleotide_length": "cdr3_rearrangement_length",
        "v_substitution_count": "v_shm_count",
        "v_substitution_indexes": "v_shm_indexes",
    }
    rename_map = {k: v for k, v in rename_map.items() if k in df.columns}
    df = df.rename(columns=rename_map)

    out = pd.DataFrame()
    for col in v1_columns:
        if col == "sample_name":
            out[col] = sample_name
        elif col == "seq_reads" and "seq_reads" not in df.columns:
            out[col] = df["templates"] if "templates" in df.columns else pd.NA
        elif col == "rearrangement_trunc" and "rearrangement_trunc" not in df.columns:
            out[col] = df.get("rearrangement", pd.NA)
        elif col in df.columns:
            out[col] = df[col].values
        else:
            out[col] = pd.NA

    if "templates" in df.columns:
        out["total_templates"] = df["templates"].sum()
        if "frame_type" in df.columns:
            prod = df["frame_type"] == "In"
            out["productive_templates"] = df.loc[prod, "templates"].sum()
            out["total_rearrangements"] = len(df)
            out["productive_rearrangements"] = prod.sum()

    out.to_csv(output_path, sep='\t', index=False)
    print(f"Done: {input_path} -> {output_path}")
    print(f"  {len(out)} rows x {len(out.columns)} columns")

# --- Run it ---
#convert_v2_to_v1("/Users/lingtingshi/Documents/Annotated_v4/Patient_154_nogood/R154_MLRCFSElo_CD4.tsv", "/Users/lingtingshi/Documents/Annotated_v4/Patient_154")

In [ ]:
import glob

input_dir = "/Users/lingtingshi/Documents/GVHD_project/Annotated_v4/Patient_154"
output_dir = "/Users/lingtingshi/Documents/Annotated_v4/Patient_154"

os.makedirs(output_dir, exist_ok=True)

for f in glob.glob(os.path.join(input_dir, "*.tsv")):
    out_path = os.path.join(output_dir, os.path.basename(f))
    convert_v2_to_v1(f, out_path)

In [8]:
import glob

input_dir = "/Users/lingtingshi/Documents/GVHD_project/Annotated_v4/Patient_176"
output_dir = "/Users/lingtingshi/Documents//Annotated_v4/Patient_176"

os.makedirs(output_dir, exist_ok=True)

for f in glob.glob(os.path.join(input_dir, "*.tsv")):
    out_path = os.path.join(output_dir, os.path.basename(f))
    convert_v2_to_v1(f, out_path)

/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[col] = df[col].values
/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[col] = df[col].values
/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performa

Done: /Users/lingtingshi/Documents/GVHD_project/Annotated_v4/Patient_176/R176_MLRCFSElo_CD4.tsv -> /Users/lingtingshi/Documents//Annotated_v4/Patient_176/R176_MLRCFSElo_CD4.tsv
  42 rows x 118 columns


/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[col] = df[col].values
/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[col] = df[col].values
/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performa

Done: /Users/lingtingshi/Documents/GVHD_project/Annotated_v4/Patient_176/R176_3_both.tsv -> /Users/lingtingshi/Documents//Annotated_v4/Patient_176/R176_3_both.tsv
  24231 rows x 118 columns


/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[col] = df[col].values
/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[col] = df[col].values
/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performa

...

/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[col] = df[col].values
/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[col] = df[col].values
/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performa

Done: /Users/lingtingshi/Documents/GVHD_project/Annotated_v4/Patient_176/R176_14_both.tsv -> /Users/lingtingshi/Documents//Annotated_v4/Patient_176/R176_14_both.tsv
  6275 rows x 118 columns


/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[col] = df[col].values
/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[col] = df[col].values
/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performa

Done: /Users/lingtingshi/Documents/GVHD_project/Annotated_v4/Patient_176/R176_unstimulated_CD8.tsv -> /Users/lingtingshi/Documents//Annotated_v4/Patient_176/R176_unstimulated_CD8.tsv
  12168 rows x 118 columns
Done: /Users/lingtingshi/Documents/GVHD_project/Annotated_v4/Patient_176/R176_MLRCFSElo_CD8.tsv -> /Users/lingtingshi/Documents//Annotated_v4/Patient_176/R176_MLRCFSElo_CD8.tsv
  12 rows x 118 columns


/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[col] = df[col].values
/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[col] = df[col].values
/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performa

In [9]:
import glob

input_dir = "/Users/lingtingshi/Documents/GVHD_project/Annotated_v4/Patient_155"
output_dir = "/Users/lingtingshi/Documents//Annotated_v4/Patient_155"

os.makedirs(output_dir, exist_ok=True)

for f in glob.glob(os.path.join(input_dir, "*.tsv")):
    out_path = os.path.join(output_dir, os.path.basename(f))
    convert_v2_to_v1(f, out_path)

/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[col] = df[col].values
/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[col] = df[col].values
/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performa

Done: /Users/lingtingshi/Documents/GVHD_project/Annotated_v4/Patient_155/R155_unstimulated_CD8.tsv -> /Users/lingtingshi/Documents//Annotated_v4/Patient_155/R155_unstimulated_CD8.tsv
  184147 rows x 118 columns


/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[col] = df[col].values
/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[col] = df[col].values
/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performa

Done: /Users/lingtingshi/Documents/GVHD_project/Annotated_v4/Patient_155/R155_3_both.tsv -> /Users/lingtingshi/Documents//Annotated_v4/Patient_155/R155_3_both.tsv
  21543 rows x 118 columns
Done: /Users/lingtingshi/Documents/GVHD_project/Annotated_v4/Patient_155/R155_MLRCFSElo_CD8.tsv -> /Users/lingtingshi/Documents//Annotated_v4/Patient_155/R155_MLRCFSElo_CD8.tsv
  1095 rows x 118 columns


/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[col] = df[col].values
/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[col] = df[col].values
/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performa

Done: /Users/lingtingshi/Documents/GVHD_project/Annotated_v4/Patient_155/R155_MLRCFSElo_CD4.tsv -> /Users/lingtingshi/Documents//Annotated_v4/Patient_155/R155_MLRCFSElo_CD4.tsv
  6779 rows x 118 columns


/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[col] = df[col].values
/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[col] = df[col].values
/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performa

Done: /Users/lingtingshi/Documents/GVHD_project/Annotated_v4/Patient_155/R155_14_both.tsv -> /Users/lingtingshi/Documents//Annotated_v4/Patient_155/R155_14_both.tsv
  229963 rows x 118 columns


/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[col] = df[col].values
/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[col] = df[col].values
/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performa

Done: /Users/lingtingshi/Documents/GVHD_project/Annotated_v4/Patient_155/R155_unstimulated_CD4.tsv -> /Users/lingtingshi/Documents//Annotated_v4/Patient_155/R155_unstimulated_CD4.tsv
  503066 rows x 118 columns


In [10]:
import glob

input_dir = "/Users/lingtingshi/Documents/GVHD_project/Annotated_v4/Patient_183"
output_dir = "/Users/lingtingshi/Documents//Annotated_v4/Patient_183"

os.makedirs(output_dir, exist_ok=True)

for f in glob.glob(os.path.join(input_dir, "*.tsv")):
    out_path = os.path.join(output_dir, os.path.basename(f))
    convert_v2_to_v1(f, out_path)

/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[col] = df[col].values
/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[col] = df[col].values
/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performa

Done: /Users/lingtingshi/Documents/GVHD_project/Annotated_v4/Patient_183/R183_unstimulated_CD8.tsv -> /Users/lingtingshi/Documents//Annotated_v4/Patient_183/R183_unstimulated_CD8.tsv
  47424 rows x 118 columns
Done: /Users/lingtingshi/Documents/GVHD_project/Annotated_v4/Patient_183/R183_MLRCFSElo_CD8.tsv -> /Users/lingtingshi/Documents//Annotated_v4/Patient_183/R183_MLRCFSElo_CD8.tsv
  152 rows x 118 columns


/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[col] = df[col].values
/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[col] = df[col].values
/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performa

Done: /Users/lingtingshi/Documents/GVHD_project/Annotated_v4/Patient_183/R183_3_both.tsv -> /Users/lingtingshi/Documents//Annotated_v4/Patient_183/R183_3_both.tsv
  15061 rows x 118 columns
Done: /Users/lingtingshi/Documents/GVHD_project/Annotated_v4/Patient_183/R183_MLRCFSElo_CD4.tsv -> /Users/lingtingshi/Documents//Annotated_v4/Patient_183/R183_MLRCFSElo_CD4.tsv
  333 rows x 118 columns


/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[col] = df[col].values
/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[col] = df[col].values
/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performa

Done: /Users/lingtingshi/Documents/GVHD_project/Annotated_v4/Patient_183/R183_14_both.tsv -> /Users/lingtingshi/Documents//Annotated_v4/Patient_183/R183_14_both.tsv
  14401 rows x 118 columns


/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[col] = df[col].values
/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[col] = df[col].values
/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performa

Done: /Users/lingtingshi/Documents/GVHD_project/Annotated_v4/Patient_183/R183_unstimulated_CD4.tsv -> /Users/lingtingshi/Documents//Annotated_v4/Patient_183/R183_unstimulated_CD4.tsv
  55391 rows x 118 columns


In [11]:
import glob

input_dir = "/Users/lingtingshi/Documents/GVHD_project/Annotated_v4/Patient_181"
output_dir = "/Users/lingtingshi/Documents//Annotated_v4/Patient_181"

os.makedirs(output_dir, exist_ok=True)

for f in glob.glob(os.path.join(input_dir, "*.tsv")):
    out_path = os.path.join(output_dir, os.path.basename(f))
    convert_v2_to_v1(f, out_path)

/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[col] = df[col].values
/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[col] = df[col].values
/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performa

Done: /Users/lingtingshi/Documents/GVHD_project/Annotated_v4/Patient_181/R181_14_both.tsv -> /Users/lingtingshi/Documents//Annotated_v4/Patient_181/R181_14_both.tsv
  32007 rows x 118 columns


/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[col] = df[col].values
/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[col] = df[col].values
/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performa

Done: /Users/lingtingshi/Documents/GVHD_project/Annotated_v4/Patient_181/R181_3_both.tsv -> /Users/lingtingshi/Documents//Annotated_v4/Patient_181/R181_3_both.tsv
  135794 rows x 118 columns


/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[col] = df[col].values
/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[col] = df[col].values
/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performa

Done: /Users/lingtingshi/Documents/GVHD_project/Annotated_v4/Patient_181/R181_unstimulated_CD4.tsv -> /Users/lingtingshi/Documents//Annotated_v4/Patient_181/R181_unstimulated_CD4.tsv
  83005 rows x 118 columns
Done: /Users/lingtingshi/Documents/GVHD_project/Annotated_v4/Patient_181/R181_MLRCFSElo_CD8.tsv -> /Users/lingtingshi/Documents//Annotated_v4/Patient_181/R181_MLRCFSElo_CD8.tsv
  11 rows x 118 columns


/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[col] = df[col].values
/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[col] = df[col].values
/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performa

Done: /Users/lingtingshi/Documents/GVHD_project/Annotated_v4/Patient_181/R181_unstimulated_CD8.tsv -> /Users/lingtingshi/Documents//Annotated_v4/Patient_181/R181_unstimulated_CD8.tsv
  36737 rows x 118 columns
Done: /Users/lingtingshi/Documents/GVHD_project/Annotated_v4/Patient_181/R181_MLRCFSElo_CD4.tsv -> /Users/lingtingshi/Documents//Annotated_v4/Patient_181/R181_MLRCFSElo_CD4.tsv
  93 rows x 118 columns


/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[col] = df[col].values
/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[col] = df[col].values
/var/folders/xw/s33dz4j90lq0syx3v499g4j00000gn/T/ipykernel_51568/3467651170.py:78: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performa